# Práctica 7 - Train, Validation y Test
**Sistemas de Aprendizaje Automático · Sesión 7**   
**Archivo de datos:** `clientes_ecommerce.csv`

### Contexto
La empresa de comercio electrónico quiere predecir si un cliente comprará durante el próximo mes. En esta práctica separarás correctamente los datos, evitarás fugas de información y entrenarás un primer clasificador **sin utilizar el conjunto de test para evaluarlo**.

**Instrucciones:** completa las celdas con `TODO`, ejecuta el notebook en orden y contesta en Markdown las preguntas. No cambies la semilla `random_state=42`. El archivo CSV debe estar en la misma carpeta que el notebook.

**Objetos finales:** `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`, `preprocesador`, `modelo`, `predicciones_train`, `predicciones_val`, `accuracy_train` y `accuracy_val`.

## Parte 1 - Preparación del dataset
### Ejercicio 1. Cargar y explorar
Carga los datos, muestra las primeras cinco filas, las dimensiones y los tipos de datos. Comprueba que el CSV se encuentra junto al notebook.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("clientes_ecommerce.csv")
print(df.head(5))
print(df.shape)
print(df.dtypes)

Responde: ¿Cuántas muestras y columnas hay? ¿Qué variable queremos predecir? ¿Es clasificación o regresión?

Respuesta: 400 muestras y 12 columnas. Queremos predecir `compra_proximo_mes`: es clasificación binaria.

### Ejercicio 2. Separar características y objetivo
Crea `X` excluyendo `cliente_id` y `compra_proximo_mes`. Crea `y` codificando `No` → 0 y `Sí` → 1. Comprueba dimensiones, valores ausentes en `y` y distribución de clases.

In [ ]:
X = df.drop(columns=["cliente_id", "compra_proximo_mes"])
y = df["compra_proximo_mes"].map({"No": 0, "Sí": 1})
print("X:", X.shape, "| y:", y.shape)
print("Ausentes en y:", y.isna().sum())
print(y.value_counts(normalize=True).round(3))

Responde: ¿Cuántas características hay? ¿Por qué debemos excluir el identificador y el target de `X`?

Respuesta: 10 características. Se excluye `cliente_id` (un identificador sin valor predictivo) y `compra_proximo_mes` (el objetivo, que sería una trampa para el modelo).

## Parte 2 - División de datos
### Ejercicio 3. Reservar test
Reserva el 20 % para test mediante `train_test_split` con `random_state=42` y `stratify=y`. El 80 % restante se llamará `X_temp`, `y_temp`.

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("X_temp:", X_temp.shape, "| X_test:", X_test.shape)

Responde: ¿Para qué sirven `random_state` y `stratify`?

Respuesta: `random_state` fija la semilla para que la división sea reproducible. `stratify` mantiene las mismas proporciones de clases en cada conjunto.

### Ejercicio 4. Obtener validación
Divide `X_temp`, `y_temp` para que el reparto final sea **60 % train / 20 % validation / 20 % test**. Utiliza `random_state=42` y estratificación.

In [ ]:
# 0.25 de 80 % = 20 % del total
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)
print("Comprobación: 0.25 * 0.8 =", 0.25 * 0.8)

Responde: ¿Qué valor de `test_size` necesitas en la segunda división? Demuéstralo con una operación.

Respuesta: `test_size=0.25`, porque 0,25 × 0,8 = 0,2 del total.

### Ejercicio 5. Comprobar dimensiones
Muestra `shape` de los seis conjuntos. Verifica que la suma de muestras coincide con el tamaño original. Completa: Train ___ muestras; Validation ___; Test ___.

In [ ]:
for nombre, conjunto in [("X_train", X_train), ("X_val", X_val), ("X_test", X_test),
                        ("y_train", y_train), ("y_val", y_val), ("y_test", y_test)]:
    print(nombre, conjunto.shape)
total = len(X_train) + len(X_val) + len(X_test)
print("Suma de muestras:", total, "| original:", len(X), "| coincide:", total == len(X))

## Parte 3 - Verificar los conjuntos
### Ejercicio 6. Distribución de clases
Calcula el porcentaje de las clases 0 y 1 en train, validation y test con `value_counts(normalize=True)`. Crea una tabla comparativa.

In [ ]:
tabla = pd.DataFrame({
    "Train": y_train.value_counts(normalize=True),
    "Validation": y_val.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True),
}).round(3)
print(tabla)

Responde: ¿Se conservan aproximadamente las proporciones? ¿Por qué importa cuando una clase es minoritaria?

Respuesta: Sí, las proporciones se conservan (unos 61 % / 39 %). Importa con clases minoritarias: una división sin estratificar podría dejar muy pocas muestras de esa clase en un conjunto.

### Ejercicio 7. Independencia de conjuntos
Comprueba con intersecciones de índices que ninguna fila original está presente en dos conjuntos. Usa `assert`. Recuerda que esto **no** detecta automáticamente duplicados con índices distintos.

In [ ]:
assert len(X_train.index.intersection(X_val.index)) == 0
assert len(X_train.index.intersection(X_test.index)) == 0
assert len(X_val.index.intersection(X_test.index)) == 0
print("Sin solapamiento de índices entre train, validation y test")

Responde: ¿Qué problema habría si se compartieran muestras? ¿Qué limitación tiene esta comprobación?

Respuesta: Si se compartieran muestras, las métricas serían demasiado optimistas. Limitación: solo detecta solapamiento de índices; no detecta duplicados con índices distintos.

## Parte 4 - Data leakage y preprocesamiento
### Ejercicio 8. Detectar errores
Analiza estos casos **sin ejecutarlos**:

**A:** `scaler.fit_transform(X)` antes de dividir en train/test.

**B:** `imputer.fit_transform(X_train)` y `imputer.transform(X_val)`.

**C:** evaluar repetidamente en `X_test` y elegir los hiperparámetros con esa puntuación.

Para cada caso explica si es correcto, cuál es el riesgo y cómo lo corregirías.

Caso A: Incorrecto. El escalador aprende media y desviación de todo el dataset, incluido test (data leakage). Corrección: ajustarlo solo con train, dentro de un `Pipeline`.

Caso B: Correcto. El imputador aprende la mediana solo de train y se aplica a validation sin reajustarlo.

Caso C: Incorrecto. Elegir hiperparámetros con test lo contamina y deja de ser independiente. Corrección: elegirlos con validation y usar test solo al final.

### Ejercicio 9. Edad anómala
Para este ejercicio, edades superiores a 120 se consideran inválidas. Sustitúyelas por `NaN` en **copias** de train, validation y test. Muestra el número de ausentes en `edad` de cada conjunto. No calcules la mediana: lo hará el pipeline.

In [ ]:
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

for datos in [X_train, X_val, X_test]:
    datos.loc[datos["edad"] > 120, "edad"] = np.nan

print("Ausentes en edad -> train:", X_train["edad"].isna().sum(),
      "| validation:", X_val["edad"].isna().sum(),
      "| test:", X_test["edad"].isna().sum())

### Ejercicio 10. Preprocesamiento
Construye tres pipelines y combínalos en `preprocesador`:

- **Numéricas:** imputación mediana + `StandardScaler`.
- **Nominales:** imputación moda + `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`.
- **Ordinal:** imputación moda + `OrdinalEncoder`, orden Bronce, Plata, Oro, Platino y desconocidos `-1`.

**No ajustes el preprocesador por separado con validation ni test.**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales", "gasto_mensual",
    "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]

pipeline_numerico = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
pipeline_nominal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
pipeline_ordinal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[["Bronce", "Plata", "Oro", "Platino"]],
                               handle_unknown="use_encoded_value", unknown_value=-1)),
])
preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_numerico, numericas),
    ("nom", pipeline_nominal, nominales),
    ("ord", pipeline_ordinal, ordinales),
])

Responde: ¿Qué ventajas tiene mantener todas las transformaciones en un único preprocesador?

Respuesta: Garantiza que cada transformación se aprende solo con train y se aplica igual a validation y test, evitando fugas de información. Además, el código es más limpio y reproducible.

## Parte 5 - Primer modelo
### Ejercicio 11. Pipeline completo
Construye `modelo` con dos etapas: `preprocesamiento` (el objeto anterior) y `clasificador` (`LogisticRegression(max_iter=1000)`).

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(max_iter=1000)),
])

Responde: ¿Qué ocurre antes de entrenar el clasificador? ¿Cómo se tratará una categoría nominal nueva?

Respuesta: Antes de entrenar, el preprocesador imputa, escala y codifica los datos. Una categoría nueva se codifica como ceros con `handle_unknown="ignore"`.

### Ejercicio 12. Entrenar
Entrena el pipeline **solo** con `X_train` e `y_train`.

In [ ]:
modelo.fit(X_train, y_train)

Responde: ¿Qué aprende el preprocesador? ¿Qué aprende el clasificador?

Respuesta: El preprocesador aprende medianas, medias, desviaciones y categorías de train. El clasificador aprende los coeficientes de la regresión logística.

### Ejercicio 13. Predecir
Crea `predicciones_train` y `predicciones_val` usando `modelo.predict()`. Muestra las primeras diez predicciones de validación. **No uses `X_test`**.

In [ ]:
predicciones_train = modelo.predict(X_train)
predicciones_val = modelo.predict(X_val)
print(predicciones_val[:10])

Responde: ¿Qué significan 0 y 1? ¿Por qué no hay que transformar manualmente `X_val` antes de `predict()`?

Respuesta: 0 = no compra y 1 = compra. No hay que transformar `X_val` a mano: el pipeline aplica el mismo preprocesado aprendido con train.

### Ejercicio 14. Accuracy
Calcula `accuracy_train` y `accuracy_val`. Muestra una tabla con ambos resultados. No se permite calcular accuracy sobre test.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_train = accuracy_score(y_train, predicciones_train)
accuracy_val = accuracy_score(y_val, predicciones_val)
print(pd.DataFrame({"Conjunto": ["Train", "Validation"],
                    "Accuracy": [accuracy_train, accuracy_val]}).round(3))

Responde: ¿Qué conjunto obtiene mayor accuracy? ¿Hay diferencia importante? ¿Una accuracy alta garantiza que el modelo sea bueno?

Respuesta: Train tiene mayor accuracy (0,708 frente a 0,662), una diferencia moderada. Una accuracy alta no garantiza un buen modelo: si siempre se predice «No», ya se acierta el 61 % en validation.

### Ejercicio 15. Comparar predicciones
Crea un DataFrame `resultados` con las columnas `Real`, `Predicción` y `Correcta`. Muestra las primeras 15 filas y cuenta aciertos y errores.

In [ ]:
resultados = pd.DataFrame({"Real": y_val.values, "Predicción": predicciones_val})
resultados["Correcta"] = resultados["Real"] == resultados["Predicción"]
print(resultados.head(15))
print(resultados["Correcta"].value_counts())

Responde: ¿Qué información adicional necesitaríamos para analizar mejor los errores del clasificador?

Respuesta: Matriz de confusión, precisión y recall por clase, y el perfil de los clientes mal clasificados (nivel de fidelidad, gasto, etc.).

## Parte 6 - Interpretación y conclusiones
### Ejercicio 16. Analizar resultados hipotéticos
| Modelo | Accuracy train | Accuracy validation |
|---|---:|---:|
| A | 0.99 | 0.61 |
| B | 0.83 | 0.81 |
| C | 0.55 | 0.53 |

Responde: ¿Cuál muestra indicios de sobreajuste? ¿Cuál es más consistente? ¿Cuál podría estar infraajustado? ¿Elegirías el de mayor accuracy de entrenamiento? ¿Qué comprobarías antes de decidir?

Respuesta: A muestra sobreajuste (0,99 en train frente a 0,61 en validación). B es el más consistente (0,83 y 0,81). C está infraajustado (0,55 y 0,53). No elegiría A solo por su accuracy de entrenamiento: elegiría B. Antes de decidir, comprobaría los resultados con validación cruzada y revisaría la matriz de errores.